In [3]:
# Author: Tam Wan Jin
from pyspark.sql import SparkSession
from pyspark.sql import Row
from auradb import *
from neo4j import GraphDatabase

URI = NEO4J_URI
USERNAME = NEO4J_USERNAME
PASSWORD = NEO4J_PASSWORD

spark = SparkSession.builder \
    .appName("FraudDetectionQueries") \
    .getOrCreate()

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD),
    connection_timeout=30,
    max_connection_lifetime=1000,
    keep_alive=True
)
import json
from pyspark.sql.types import *

def query_potential_fraud(tx):
    query = """
    MATCH (ip:IP)<-[:USES_IP]-(a:SenderAccount)
    RETURN ip.address AS IP, 
           count(a) AS Num_Accounts,
           collect(a.id) AS Account_List
    ORDER BY Num_Accounts DESC
    """
    result = tx.run(query)
    return [record.data() for record in result]

with driver.session() as session:
    data = session.execute_read(query_potential_fraud)
    
schema = StructType([
    StructField("IP", StringType(), True),
    StructField("Num_Accounts", IntegerType(), True),
    StructField("Account_List", StringType(), True)
])

for r in data:
    r["Account_List"] = json.dumps(r["Account_List"])

print("Query 2: IP Takeover Accounts")
df1 = spark.createDataFrame(data, schema=schema)
df1.show(truncate=False)

def query_fraud_senders(tx):
    query = """
    MATCH (s:SenderAccount)-[t:TRANSFER]->(r:ReceiverAccount:SuspiciousReceiver)
    RETURN
        r.id AS Suspicious_Receiver,
        count(t) AS Receive_Count,
        sum(toFloat(t.amount)) AS Total_Amount_Received,
        collect(DISTINCT s.id) AS  Sender_List
    ORDER BY Total_Amount_Received DESC
    """
    result = tx.run(query)
    return [record.data() for record in result]

with driver.session() as session:
    data = session.execute_read(query_fraud_senders)

schema = StructType([
    StructField("Suspicious_Receiver", StringType(), True),
    StructField("Receive_Count", IntegerType(), True),
    StructField("Total_Amount_Received", DoubleType(), True),
    StructField("Sender_List", ArrayType(StringType()), True),
])
print("Query 1: Suspicious Receiver with Extreme High Amount")
df = spark.createDataFrame(data, schema=schema)
df.show(truncate=False)



Query 2: IP Takeover Accounts
+---------------+------------+------------------------------------------------------------------------------+
|IP             |Num_Accounts|Account_List                                                                  |
+---------------+------------+------------------------------------------------------------------------------+
|73.28.186.0    |6           |["ACC429409", "ACC852524", "ACC486667", "ACC323048", "ACC250631", "ACC475503"]|
|75.61.19.32    |5           |["ACC965251", "ACC957763", "ACC912772", "ACC310412", "ACC139671"]             |
|215.183.135.237|5           |["ACC716966", "ACC737794", "ACC663308", "ACC662826", "ACC333167"]             |
|38.14.180.243  |4           |["ACC577498", "ACC852437", "ACC615339", "ACC367151"]                          |
|141.52.142.173 |4           |["ACC154034", "ACC253114", "ACC819283", "ACC816328"]                          |
|159.208.79.136 |4           |["ACC127871", "ACC550834", "ACC519298", "ACC555316"]        